# 08 - Troubleshooting, etcd Simulations, and Advanced Ops

Final notebook for advanced operations on your local stack.

Coverage:
- health matrix and restart/error scanning
- memory pressure analysis and footprint reduction hints
- dry-run Helm rendering for optional components (`etcd`, seeder CronJob)
- one-shot observability seeding patterns


In [ ]:
import os
import sys
from pathlib import Path

expected = "/usr/local/bin/python3.11"
print(f"Python executable: {sys.executable}")
print(f"Python version   : {sys.version.split()[0]}")
if Path(sys.executable).resolve().as_posix() != expected:
    print(f"WARNING: This notebook is designed for {expected}.")
    print("Switch the notebook kernel to Python 3.11 before continuing.")
else:
    print("Kernel check passed.")


In [ ]:
import importlib
import subprocess
import sys

packages = ["requests", "pandas", "matplotlib", "pyyaml", "langsmith", "tabulate"]
missing = []
for pkg in packages:
    module_name = "yaml" if pkg == "pyyaml" else pkg
    try:
        importlib.import_module(module_name)
    except Exception:
        missing.append(pkg)

if missing:
    print("Installing missing packages:", ", ".join(missing))
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *missing])
else:
    print("All common packages are already installed.")


In [ ]:
import json
import os
import shlex
import subprocess
from pathlib import Path


def run(cmd: str, check: bool = True):
    print(f"$ {cmd}")
    proc = subprocess.run(cmd, shell=True, text=True, capture_output=True)
    if proc.stdout:
        print(proc.stdout.rstrip())
    if proc.stderr:
        print(proc.stderr.rstrip())
    if check and proc.returncode != 0:
        raise RuntimeError(f"Command failed ({proc.returncode}): {cmd}")
    return proc


def run_json(cmd: str):
    proc = run(cmd)
    return json.loads(proc.stdout)


PROJECT_ROOT = Path("/media/waqasm86/External1/Project-Nvidia-Office/Project-Llamatelemetry/langchain-kubernetes-jupyterlab/llm-observability-stack")
NAMESPACE = "llm-observability"
print(f"Project root: {PROJECT_ROOT}")


## Current Cluster Health Matrix


In [ ]:
pods = run_json(f"kubectl get pods -n {NAMESPACE} -o json")
rows = []
for item in pods.get("items", []):
    status = item.get("status", {})
    container_statuses = status.get("containerStatuses", [])
    restarts = sum(cs.get("restartCount", 0) for cs in container_statuses)
    rows.append(
        {
            "pod": item["metadata"]["name"],
            "phase": status.get("phase"),
            "ready": sum(1 for cs in container_statuses if cs.get("ready")),
            "containers": len(container_statuses),
            "restarts": restarts,
            "node": status.get("nodeName"),
        }
    )

import pandas as pd
health_df = pd.DataFrame(rows).sort_values(["phase", "restarts", "pod"], ascending=[True, False, True])
health_df


## Recent Warning Events


In [ ]:
run(f"kubectl get events -n {NAMESPACE} --sort-by=.metadata.creationTimestamp | tail -n 40", check=False)


## Memory Pressure Ranking


In [ ]:
import matplotlib.pyplot as plt

raw = run(f"kubectl top pods -n {NAMESPACE} --no-headers", check=False).stdout
rows = []
for line in raw.splitlines():
    parts = line.split()
    if len(parts) >= 3:
        mem = parts[2]
        mem_mi = None
        if mem.endswith("Mi"):
            mem_mi = float(mem[:-2])
        elif mem.endswith("Gi"):
            mem_mi = float(mem[:-2]) * 1024
        rows.append({"pod": parts[0], "cpu": parts[1], "memory": mem, "memory_mi": mem_mi})

mem_df = pd.DataFrame(rows).sort_values("memory_mi", ascending=False)
display(mem_df)

if not mem_df.empty:
    ax = mem_df.head(10).plot(x="pod", y="memory_mi", kind="bar", figsize=(11, 4), title="Top Memory Pods")
    ax.set_ylabel("Memory (Mi)")
    plt.xticks(rotation=45, ha="right")
    plt.show()


## Auto-Generate Footprint Reduction Suggestions


In [ ]:
import yaml

values = yaml.safe_load((PROJECT_ROOT / "values.local-k3s.yaml").read_text())
suggestions = []

if values.get("pythonToolbox", {}).get("enabled"):
    suggestions.append("Disable pythonToolbox when not debugging: --set pythonToolbox.enabled=false")
if values.get("langsmithDashboardSeeder", {}).get("enabled"):
    suggestions.append("Disable continuous seeding if not needed: --set langsmithDashboardSeeder.enabled=false")
if values.get("etcd", {}).get("enabled"):
    suggestions.append("Disable etcd unless testing failure drills: --set etcd.enabled=false")

if values.get("ollama", {}).get("service", {}).get("type") == "LoadBalancer":
    suggestions.append("Set ollama service to ClusterIP in local profile to reduce svclb overhead")
if values.get("langchainDemo", {}).get("service", {}).get("type") == "LoadBalancer":
    suggestions.append("Set langchain-demo service to ClusterIP to remove extra svclb pod")

if not suggestions:
    suggestions.append("Current values already follow a low-footprint local profile.")

for idx, item in enumerate(suggestions, 1):
    print(f"{idx}. {item}")


## Helm Render Checks for Optional Components


In [ ]:
run(f"cd {PROJECT_ROOT} && helm template llm-observability-stack . -f values.local-k3s.yaml > /tmp/rendered-current.yaml")
run("rg -n 'kind: CronJob|name: langsmith-dashboard-seeder|kind: StatefulSet|app.kubernetes.io/name: etcd' /tmp/rendered-current.yaml", check=False)

run(f"cd {PROJECT_ROOT} && helm template llm-observability-stack . -f values.local-k3s.yaml --set etcd.enabled=true > /tmp/rendered-etcd.yaml")
run("rg -n 'app.kubernetes.io/name: etcd|kind: StatefulSet' /tmp/rendered-etcd.yaml", check=False)


## Optional One-Shot Seeder Execution Pattern


In [ ]:
RUN_ONE_SHOT_SEED = False

if RUN_ONE_SHOT_SEED:
    run(
        "kubectl create job -n llm-observability --from=cronjob/langsmith-dashboard-seeder langsmith-dashboard-seeder-manual-$(date +%s)",
        check=False,
    )
else:
    print("Dry-run mode. To execute, set RUN_ONE_SHOT_SEED=True and ensure the CronJob exists.")


## End-to-End Validation Commands


In [ ]:
commands = [
    "kubectl get pods -A",
    "kubectl get svc -n llm-observability",
    "kubectl logs -n llm-observability deploy/langchain-demo --tail=50",
    "nvidia-smi",
]
for cmd in commands:
    run(cmd, check=False)


## Tutorial Series Complete
You now have an end-to-end, production-style notebook flow from environment setup through advanced troubleshooting.
